In [ ]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 3 – Location problems
#  Section: Exercise 3
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP        # Modeling language
using HiGHS       # Solver
using CSV         # For reading CSV files
using DataFrames  # For handling DataFrame operations

# Include utility functions for plotting and distance matrix creation
include("utils/sclp_utils.jl")

# Main function to solve SCLP
function solve_sclp(file_path; radius = 200)
    # Load coordinates
    coordinates = CSV.read(file_path, header=true, DataFrame) |> Matrix{Float64}
    
    # Create distance matrix
    D = create_distance_matrix(coordinates)
    
    # Create coverage matrix considering the defined radius
    C = D .< radius

    # Number of demand/facility points
    n = size(C, 1)

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Variables
    @variable(model, x[1:n], Bin)

    # Objective: minimize number of facilities opened
    @objective(model, Min, sum(x))

    # Constraint: each demand point is covered by at least one facility
    @constraint(model, [i in 1:n], sum(C[i,j] * x[j] for j in 1:n) >= 1)

    # Solve the model
    JuMP.optimize!(model)

    # Get solution vector
    x_vals = JuMP.value.(x)

    # Get indices of selected facilities
    selected_facilities = findall(xi -> xi > 0.5, x_vals)

    # Display solution
    println("Number of facilities opened: ", JuMP.objective_value(model))
    println("Selected facility indices: ", selected_facilities)

    # Display solution on map
    map = plot_solution(selected_facilities, coordinates, radius)
    display(map)
end

# Example usage
solve_sclp("data/sjc_coordinates.csv", radius = 400)

Number of facilities opened: 4.0
Selected facility indices: [1, 43, 50, 65]


PyObject <folium.folium.Map object at 0x000001D813D49730>